# 02 - Training and Evaluation

This notebook covers the training and evaluation phase of the Neural Machine Translation (NMT) project.
It loads the preprocessed data and vocabularies, initializes the sequence-to-sequence model (Encoder-Decoder with Attention), and runs the training loop using Teacher Forcing.

It also includes placeholders for optional tasks such as BLEU score evaluation and Beam Search decoding.

In [1]:
import os
import sys
import random
import math
import time

sys.path.append(os.path.abspath(os.path.join('..','src')))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Import our custom modules
from vocabulary import Vocabulary
from dataset import TranslationDataset, collate_fn, load_processed
from encoder import Encoder
from decoder import Decoder
from seq2seq import Seq2Seq
from transformer import TransformerSeq2Seq

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

PyTorch version: 2.8.0+cu128
Using device: cuda


## 1. Configuration & Hyperparameters
Define the sizes of the neural network layers, learning rate, and batch size. This is where you can perform hyperparameter tuning.

In [2]:
data_dir = os.path.abspath(os.path.join('..','data','processed'))
models_dir = os.path.abspath(os.path.join('..','models'))

os.makedirs(models_dir, exist_ok=True)

# Choose which model to train/evaluate: 'RNN' or 'TRANSFORMER'.
# Everything downstream (data loading, training loop, BLEU) is shared -- only
# the model-construction cell branches on this flag.
MODEL_TYPE = 'TRANSFORMER'

# --- Shared hyperparameters ---
BATCH_SIZE = 64
N_EPOCHS = 10
CLIP = 1.0

# --- RNN hyperparameters (used when MODEL_TYPE == 'RNN') ---
ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
ENC_HID_DIM = 256
DEC_HID_DIM = 256
ATTN_DIM = 256
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

# --- Transformer hyperparameters (used when MODEL_TYPE == 'TRANSFORMER') ---
D_MODEL = 256      # model/embedding width (must be divisible by N_HEADS)
N_HEADS = 8        # number of attention heads
N_LAYERS = 3       # how many encoder layers and decoder layers to stack
D_FF = 512         # hidden size of the feed-forward sub-layer
TF_DROPOUT = 0.1   # transformers use lighter dropout than the RNN above

# Transformers train better with a smaller Adam learning rate than the RNN.
LEARNING_RATE = 0.001 if MODEL_TYPE == 'RNN' else 0.0005

## 2. Loading Data
Load the `.pt` files and vocabularies created in the `01_data_preparation.ipynb` notebook.

In [3]:
# Load vocabularies
src_vocab = Vocabulary.load(os.path.join(data_dir, "vocab.src.json"))
tgt_vocab = Vocabulary.load(os.path.join(data_dir, "vocab.tgt.json"))

print(f"Source vocabulary size: {len(src_vocab)}")
print(f"Target vocabulary size: {len(tgt_vocab)}")

# Load datasets
train_data = load_processed(os.path.join(data_dir, "train.pt"))
val_data = load_processed(os.path.join(data_dir, "val.pt"))
test_data = load_processed(os.path.join(data_dir, "test.pt"))

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
print(f"Testing examples: {len(test_data)}")

pad_idx = src_vocab.token2id[Vocabulary.PAD]

# Create DataLoaders
train_loader = DataLoader(TranslationDataset(train_data), batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))
val_loader = DataLoader(TranslationDataset(val_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))
test_loader = DataLoader(TranslationDataset(test_data), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))

Source vocabulary size: 29961
Target vocabulary size: 48596
Training examples: 246491
Validation examples: 13693
Testing examples: 13695


## 3. Model Initialization
Instantiate the Encoder, Decoder, and the full Seq2Seq model. We also define the Optimizer (Adam) and the Loss Function (CrossEntropyLoss).

In [4]:
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(tgt_vocab)
SOS_IDX = tgt_vocab.token2id[Vocabulary.SOS]
EOS_IDX = tgt_vocab.token2id[Vocabulary.EOS]

if MODEL_TYPE == 'RNN':
    enc = Encoder(INPUT_DIM, ENC_EMB_DIM, ENC_HID_DIM, DEC_HID_DIM, dropout=ENC_DROPOUT, pad_idx=pad_idx)
    if enc.bidirectional:
        enc_hid_dim = ENC_HID_DIM * 2
    else:
        enc_hid_dim = ENC_HID_DIM
    dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, enc_hid_dim, DEC_HID_DIM, ATTN_DIM, dropout=DEC_DROPOUT, pad_idx=pad_idx)
    model = Seq2Seq(enc, dec, device, pad_idx, SOS_IDX, EOS_IDX).to(device)
else:
    # Transformer model: same forward(src, src_lens, tgt) / translate(...) API as
    # the RNN Seq2Seq, so the training loop and evaluation below are unchanged.
    model = TransformerSeq2Seq(
        INPUT_DIM, OUTPUT_DIM, device,
        d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
        d_ff=D_FF, dropout=TF_DROPOUT,
        pad_idx=pad_idx, sos_idx=SOS_IDX, eos_idx=EOS_IDX,
    ).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

The model has 36,553,428 trainable parameters


## 4. Training Loop
Define the functions for training (with Teacher Forcing) and evaluating (without Teacher Forcing) for a single epoch.

In [5]:
from torch.amp import autocast

USE_AMP = device.type == 'cuda' and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16
print(f'AMP enabled: {USE_AMP} (dtype: {AMP_DTYPE if USE_AMP else "fp32"})')

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for batch in iterator:
        src, src_lens, tgt, _ = batch
        src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)

        optimizer.zero_grad()

        with autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            output = model(src, src_lens, tgt)
            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt_out = tgt[:, 1:].reshape(-1)
            loss = criterion(output, tgt_out)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()
    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for _, batch in enumerate(iterator):
            src, src_lens, tgt, _ = batch
            src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)

            # Turn off teacher forcing for evaluation (ratio = 0)
            output = model(src, src_lens, tgt, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]
            output = output[:, 1:].reshape(-1, output_dim)
            tgt = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

AMP enabled: True (dtype: torch.bfloat16)


Execute training

In [6]:
best_valid_loss = float('inf')
if MODEL_TYPE == 'RNN':
    model_filename = f"rnn_emb{ENC_EMB_DIM}_hid{ENC_HID_DIM}_lr{LEARNING_RATE}.pt"
else:
    model_filename = f"transformer_d{D_MODEL}_h{N_HEADS}_l{N_LAYERS}_lr{LEARNING_RATE}.pt"
model_save_path = os.path.join(models_dir, model_filename)
print(f"Starting training... Model will be saved to: {model_save_path}")
TRAINING = False

if TRAINING:
    for epoch in range(N_EPOCHS):
        start_time = time.time()
        
        train_loss = train(model, train_loader, optimizer, criterion, CLIP)
        valid_loss = evaluate(model, val_loader, criterion)
        scheduler.step(valid_loss)
        
        end_time = time.time()
        
        # Save the model if it's the best one we've seen so far
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), model_save_path)
        
        # Perplexity (PPL) is a standard metric in NLP, calculated as exp(loss)
        print(f'Epoch: {epoch+1:02} | Time: {int(end_time - start_time)}s')
        print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
        print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

Starting training... Model will be saved to: /home/jovyan/Machine_Translation/models/transformer_d256_h8_l3_lr0.0005.pt


## 5. Inference and Evaluation
Load the best model and translate sentences from the test set.

In [7]:
# Load best weights
model.load_state_dict(torch.load(model_save_path))

def strip_special(tokens):
    return [t for t in tokens if t not in (Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK)]

# Translate a few sentences from the test set
print("\n--- Example Translations ---")
model.eval()
with torch.no_grad():
    batch = next(iter(test_loader))
    src, src_lens, tgt, _ = batch
    src, src_lens, tgt = src.to(device), src_lens.to(device), tgt.to(device)
    
    # Use the greedy translation method implemented in seq2seq.py
    # TODO (Optional): Implement Beam Search in seq2seq.py and use it here instead of greedy decoding
    translations, attentions = model.translate(src, src_lens, max_len=50)
    
    for i in range(min(5, src.size(0))): # Show up to 5 examples
        src_sentence = ' '.join(strip_special(src_vocab.decode(src[i].tolist())))
        tgt_sentence = ' '.join(strip_special(tgt_vocab.decode(tgt[i].tolist())))
        pred_sentence = ' '.join(strip_special(tgt_vocab.decode(translations[i].tolist())))
        
        print(f"\nSource: {src_sentence}")
        print(f"Target: {tgt_sentence}")
        print(f"Predicted: {pred_sentence}")


--- Example Translations ---


/opt/micromamba/lib/python3.11/site-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



Source: he looks good for his age .
Target: il a l'air bien pour son âge .
Predicted: il semble bien pour son âge .

Source: he was nearly hit by the car while crossing the street .
Target: il a failli être heurté par la voiture en traversant la rue .
Predicted: il a failli se faire renverser tandis qu'il a frappé la rue .

Source: tom tried to persuade mary to stay at home .
Target: tom a essayé de persuader mary de rester chez elle .
Predicted: tom a essayé de persuader mary de rester à la maison .

Source: in spite of the rain , i went out .
Target: je suis sortie , malgré la pluie .
Predicted: malgré la pluie , je suis sortie malgré la pluie .

Source: the colonel said the situation is under control .
Target: le colonel dit que la situation est sous contrôle .
Predicted: on dit que la situation se cache sous contrôle .


In [8]:
from nltk.translate.bleu_score import corpus_bleu

def compute_bleu(model, loader, src_vocab, tgt_vocab, device, max_len=50):
    model.eval()
    hypotheses = []   # list of list of tokens
    references  = []  # list of list of list of tokens (corpus_bleu format)
    
    special = {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK}
    
    with torch.no_grad():
        for src, src_lens, tgt, _ in loader:
            src, src_lens = src.to(device), src_lens.to(device)
            translations, _ = model.translate(src, src_lens, max_len=max_len)
            
            for i in range(src.size(0)):
                pred = [t for t in tgt_vocab.decode(translations[i].tolist())
                        if t not in special]
                ref  = [t for t in tgt_vocab.decode(tgt[i].tolist())
                        if t not in special]
                hypotheses.append(pred)
                references.append([ref])   # corpus_bleu attend [[ref1], [ref2], ...]
    
    return corpus_bleu(references, hypotheses) * 100   # score en %

bleu = compute_bleu(model, test_loader, src_vocab, tgt_vocab, device)
print(f"BLEU score on test set: {bleu:.2f}")

BLEU score on test set: 43.60


In [ ]:
import time, json
from nltk.translate.bleu_score import corpus_bleu

# Each config has a "type" ("rnn" or "transformer") + its hyperparameters.
# "epochs" is optional per config (defaults to DEFAULT_EPOCHS).
DEFAULT_EPOCHS = 10
BLEU_MAX_SENTENCES = 2000   # quick BLEU on a test subset (full test set is slow)

EXPERIMENTS = [
    # --- Transformer variants (10 epochs each, ~25 min) ---
    {"name": "tf_baseline",    "type": "transformer", "d_model": 256, "n_heads": 8, "n_layers": 3, "d_ff": 512, "dropout": 0.1, "lr": 5e-4},
    {"name": "tf_deeper",      "type": "transformer", "d_model": 256, "n_heads": 8, "n_layers": 6, "d_ff": 512, "dropout": 0.1, "lr": 5e-4},
    {"name": "tf_more_drop",   "type": "transformer", "d_model": 256, "n_heads": 8, "n_layers": 3, "d_ff": 512, "dropout": 0.3, "lr": 5e-4},
    # --- RNN variants (6 epochs each; it overfits early, ~70 min each) ---
    {"name": "rnn_baseline",   "type": "rnn", "emb_dim": 128, "hid_dim": 256, "attn_dim": 256, "dropout": 0.5, "lr": 1e-3, "epochs": 6},
    {"name": "rnn_bigger",     "type": "rnn", "emb_dim": 256, "hid_dim": 512, "attn_dim": 256, "dropout": 0.5, "lr": 1e-3, "epochs": 6},
]

def build_model(cfg):
    if cfg["type"] == "rnn":
        enc = Encoder(INPUT_DIM, cfg["emb_dim"], cfg["hid_dim"], cfg["hid_dim"],
                      dropout=cfg["dropout"], pad_idx=pad_idx)
        enc_hid = cfg["hid_dim"] * 2 if enc.bidirectional else cfg["hid_dim"]
        dec = Decoder(OUTPUT_DIM, cfg["emb_dim"], enc_hid, cfg["hid_dim"], cfg["attn_dim"],
                      dropout=cfg["dropout"], pad_idx=pad_idx)
        return Seq2Seq(enc, dec, device, pad_idx, SOS_IDX, EOS_IDX).to(device)
    return TransformerSeq2Seq(
        INPUT_DIM, OUTPUT_DIM, device,
        d_model=cfg["d_model"], n_heads=cfg["n_heads"], n_layers=cfg["n_layers"],
        d_ff=cfg["d_ff"], dropout=cfg["dropout"],
        pad_idx=pad_idx, sos_idx=SOS_IDX, eos_idx=EOS_IDX).to(device)

def quick_bleu(model, data, max_sentences=BLEU_MAX_SENTENCES, max_len=50):
    special = {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK}
    loader = DataLoader(TranslationDataset(data[:max_sentences]), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=lambda b: collate_fn(b, pad_id=pad_idx))
    model.eval(); hyps, refs = [], []
    with torch.no_grad():
        for src, src_lens, tgt, _ in loader:
            src, src_lens = src.to(device), src_lens.to(device)
            trans, _ = model.translate(src, src_lens, max_len=max_len)
            for i in range(src.size(0)):
                hyps.append([t for t in tgt_vocab.decode(trans[i].tolist()) if t not in special])
                refs.append([[t for t in tgt_vocab.decode(tgt[i].tolist()) if t not in special]])
    return corpus_bleu(refs, hyps) * 100

results = []
for cfg in EXPERIMENTS:
    epochs = cfg.get("epochs", DEFAULT_EPOCHS)
    print(f"\n=== {cfg['name']} ({cfg['type']}, {epochs} epochs) === {cfg}")
    try:
        model = build_model(cfg)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        optimizer = optim.Adam(model.parameters(), lr=cfg["lr"])
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

        best_val = float("inf")
        save_path = os.path.join(models_dir, f"sweep_{cfg['name']}.pt")
        t0 = time.time()
        for epoch in range(epochs):
            tr = train(model, train_loader, optimizer, criterion, CLIP)
            vl = evaluate(model, val_loader, criterion)
            scheduler.step(vl)
            if vl < best_val:
                best_val = vl
                torch.save(model.state_dict(), save_path)
            print(f"  epoch {epoch+1}/{epochs}  train {tr:.3f}  val {vl:.3f}  ppl {math.exp(vl):.2f}")
        elapsed = time.time() - t0

        model.load_state_dict(torch.load(save_path))   # best checkpoint
        bleu = quick_bleu(model, test_data)
        results.append({**cfg, "params": n_params, "best_val_ppl": math.exp(best_val),
                        "bleu": bleu, "minutes": elapsed / 60})
        print(f"  -> BLEU(first {BLEU_MAX_SENTENCES}) {bleu:.2f} | val PPL {math.exp(best_val):.2f} | {elapsed/60:.1f} min")
    except Exception as e:
        print(f"  !! {cfg['name']} failed: {e}")
        results.append({**cfg, "params": None, "best_val_ppl": None, "bleu": None, "minutes": None})
    finally:
        try: del model
        except NameError: pass
        if device.type == "cuda":
            torch.cuda.empty_cache()

with open(os.path.join(models_dir, "sweep_results.json"), "w") as f:
    json.dump(results, f, indent=2)

print("\n\n==== SWEEP SUMMARY ====")
print(f"{'name':<16}{'type':<13}{'params':>12}{'valPPL':>9}{'BLEU':>7}{'min':>7}")
for r in results:
    if r["bleu"] is None:
        print(f"{r['name']:<16}{r['type']:<13} FAILED")
    else:
        print(f"{r['name']:<16}{r['type']:<13}{r['params']:>12,}{r['best_val_ppl']:>9.2f}{r['bleu']:>7.2f}{r['minutes']:>7.1f}")

In [9]:
# COMET score calculation
import os
os.environ["USE_TF"] = "0"

from comet import download_model, load_from_checkpoint
def compute_comet(model, loader, src_vocab, tgt_vocab, device, max_len=50):
    """
    Compute COMET score on a test set.
    import torch
import transformers
import google.protobuf

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Protobuf:", google.protobuf.__version__)
    COMET (Crosslingual Optimized Metric for Evaluation of Translation) is a neural 
    metric that correlates better with human judgment than BLEU.
    
    Args:
        model: the NMT model (Seq2Seq or Transformer)
        loader: DataLoader with test data
        src_vocab: source vocabulary
        tgt_vocab: target vocabulary
        device: torch device
        max_len: maximum translation length
        
    Returns:
        Average COMET score (0-1 range)
    """
    model.eval()
    sources = []       # list of source sentences (strings)
    hypotheses = []    # list of translated sentences (strings)
    references = []    # list of reference sentences (strings)
    
    special = {Vocabulary.PAD, Vocabulary.SOS, Vocabulary.EOS, Vocabulary.UNK}
    
    # Generate translations for all test examples
    with torch.no_grad():
        for src, src_lens, tgt, _ in loader:
            src, src_lens = src.to(device), src_lens.to(device)
            translations, _ = model.translate(src, src_lens, max_len=max_len)
            
            for i in range(src.size(0)):
                # Decode and clean each sequence
                src_tokens = [t for t in src_vocab.decode(src[i].tolist()) if t not in special]
                hyp_tokens = [t for t in tgt_vocab.decode(translations[i].tolist()) if t not in special]
                ref_tokens = [t for t in tgt_vocab.decode(tgt[i].tolist()) if t not in special]
                
                # Convert to strings
                sources.append(' '.join(src_tokens))
                hypotheses.append(' '.join(hyp_tokens))
                references.append(' '.join(ref_tokens))
    
    # Load pre-trained COMET model
    print("Downloading COMET model...")
    comet_model = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))
    
    # Prepare data in the format expected by COMET
    data = [
        {"src": src, "mt": hyp, "ref": ref}
        for src, hyp, ref in zip(sources, hypotheses, references)
    ]
    
    # Compute COMET scores
    print("Computing COMET scores...")
    scores = comet_model.predict(data, batch_size=8, gpus=1 if device.type == 'cuda' else 0)
    
    # Return average system score (0-1 range, typically higher is better)
    return scores.system_score * 100  # scale to 0-100 for easier interpretation

# Compute and display COMET score
comet = compute_comet(model, test_loader, src_vocab, tgt_vocab, device)
print(f"COMET score on test set: {comet:.2f}")

/opt/micromamba/lib/python3.11/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/opt/micromamba/lib/python3.11/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


Computing COMET scores...


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA A40') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|██████████| 1712/1712 [01:11<00:00, 23.83it/s]


COMET score on test set: 81.78
